## HDSoCv2 ROI Mode, External Trigger (debug version)

Adapted from the HDSoCv1_evalr2 ROI notebook for HDSoCv2. Structured to isolate why external-trigger runs return 0 events:

1. **Test 1** `trig=imm`, `lb=forced`: does the acquisition backend collect events at all?
2. **Test 2** `trig=ext`, `lb=forced`: does the external trigger reach the board and get accepted?
3. **Test 3** `trig=ext`, `lb=roi`: does ROI mode read out the signal channel?

Changes from the v1 notebook are marked `# CHANGED (v2)` or `# CHANGED (debug)`.

### Naludaq Version
*Tested against*: naludaq `0.36.1`, register maps from naluconfigs `18.4.1`

### Compatible Boards
+ `HDSoCv2` (`hdsocv2_evalr2`, or `hdsocv2_eval` for the earlier board revision)

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import time
import numpy as np
from copy import deepcopy
from pathlib import Path

from naludaq.board import Board, startup_board
from naludaq.communication import AnalogRegisters, DigitalRegisters, ControlRegisters
from naludaq.parsers import get_parser
from naludaq.tools import get_pedestals_controller
from naludaq.tools.pedestals.pedestals_correcter import PedestalsCorrecter

from naludaq.backend import AcquisitionManager, RemoteAcquisition

import matplotlib.pyplot as plt

In [ ]:
# import logging
#
# def setup_logger(level=logging.DEBUG):
#     """Setup a basic logger."""
#     logger = logging.getLogger()
#     handler = logging.StreamHandler()
#     handler.setFormatter(
#         logging.Formatter("%(asctime)s %(name)-30s [%(levelname)-6s]: %(message)s")
#     )
#     logger.addHandler(handler)
#     logger.setLevel(level)
#     return logger
#
# try:
#     logger.debug("logger already setup")
# except NameError:
#     logger = setup_logger()

## Function Definitions

In [ ]:
def create_acquisition(name: str, board: Board, readout_metadata: dict = None) -> RemoteAcquisition:
    """Create an acquisition, set it as the backend output, and return it."""
    am = AcquisitionManager(board)
    acquisition = am.create(name)
    acquisition.set_output()
    if readout_metadata:
        acquisition.readout_metadata = readout_metadata
    return acquisition

def parse_data(raw_data: dict, board: Board, correct_peds: bool = True) -> dict:
    """Parse an event dict that contains a 'rawdata' key, optionally pedestal-correcting it."""
    parser = get_parser(board.params)
    evt = parser.parse(raw_data)
    if correct_peds:
        PedestalsCorrecter(board.params, board.pedestals).run(evt)
    return evt

def create_acq_name(base_name: str) -> str:
    """Append a timestamp to base_name: <base_name>_<YYYY-mm-dd-HH-MM-SS>."""
    return base_name + "_" + time.strftime("%Y-%m-%d-%H-%M-%S")

def clear_figs():
    """Close all matplotlib figures."""
    plt.close("all")

def create_fig(figsize: tuple) -> plt.Figure:
    """Create a figure with constrained layout and white background."""
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    fig.set_facecolor("white")
    return fig

def plot_single_event(event, show: bool = True):
    """Plot every non-empty channel of an event to the current figure."""
    for chan, chan_data in enumerate(event["data"]):
        if len(chan_data) == 0:
            continue
        if np.all(np.isnan(chan_data)):
            continue
        plt.plot(chan_data, ".-", markersize=10, label=f"Channel {chan}")
    plt.legend(fontsize=12)
    if show:
        plt.show()

# CHANGED (debug): one function per readout test, so each test writes to its own acquisition.
def run_readout_test(label: str, readout_params: dict, use_ext_trigger: bool, readout_time: int) -> RemoteAcquisition:
    """Run one readout into a fresh acquisition and print the event count once per second.

    Args:
        label: short name, used in the acquisition name.
        readout_params: kwargs for BOARD.control.start_readout.
        use_ext_trigger: if True, the trigger generator output is switched on during the run.
        readout_time: run length in seconds.
    Returns:
        The acquisition the data was written to.
    """
    acq_name = create_acq_name(f"{BOARD.model}_{label}")
    metadata = {"params": deepcopy(PARAMS), "readout_params": deepcopy(readout_params)}
    acq = create_acquisition(acq_name, BOARD, metadata)
    acq.pedestals = BOARD.pedestals

    TRIGGER_INPUT.rf_off()
    CHANNEL_INPUT.rf_on()
    time.sleep(0.25)
    try:
        BOARD.control.start_readout(**readout_params)
        if use_ext_trigger:
            TRIGGER_INPUT.rf_on()
        for second in range(readout_time):
            time.sleep(1)
            print(f"  t = {second + 1:3d} s   events = {len(acq)}")
    finally:
        TRIGGER_INPUT.rf_off()
        CHANNEL_INPUT.rf_off()
        time.sleep(1)
        BOARD.control.stop_readout()
        time.sleep(0.25)

    print(f"{label}: {len(acq)} events in {acq_name}")
    return acq

## Directory Setup

In [ ]:
DATA_DIR = Path("./data").absolute()
DATA_DIR.mkdir(exist_ok=True)
SERVER_DIR = Path("./server").absolute()
SERVER_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("figures").absolute()
FIGURES_DIR.mkdir(exist_ok=True)

## Board Initialization

Two HDSoCv2 board definitions exist in naluconfigs: `hdsocv2_evalr2` and `hdsocv2_eval`. They differ in clock file and in the I2C addresses of the external bias DACs, so use the one that matches your board revision.

In [ ]:
BOARD_MODEL = "hdsocv2_evalr2"  # CHANGED (v2): was 'hdsocv1_evalr2'

BOARD = Board(BOARD_MODEL)
BOARD.start_server(str(SERVER_DIR))
BOARD_ADDRESS = ("192.168.1.59", 4660)
RECEIVER_ADDRESS = ("192.168.1.197", 4663)
SERVER_ADDRESS = BOARD.context.address
BOARD.connect_udp(BOARD_ADDRESS, RECEIVER_ADDRESS, SERVER_ADDRESS)
startup_board(BOARD)  # CHANGED: the second startup_board call (separate cell) was removed

## Signal Generator

### Signal Generator Settings

In [ ]:
SIGNAL_CHANNEL = 2  # CHANGED (debug): channel the sine is connected to; used for thresholds and plotting

SINE_FREQ = 1 / (4 * 32 * 1e-9)  # 4 windows -> 128 ns period, 7.8125 MHz
SINE_AMP = 0.3                   # Vpp

TRIGGER_FREQ = 1 / (1e-6)        # 1 us period within a burst
PULSE_WIDTH = 50e-9
TRIGGER_HIGH = 2.5               # V
TRIGGER_LOW = 0.0                # V

# Bursted trigger pulses
N_CYCLES = 3
BURST_PERIOD = 1                 # s

### Signal Generator Setup

**Trigger levels:** in the v1 notebook, `set_amp(2.5)` with `set_offset(0)` produces a 2.5 Vpp pulse centered on 0 V (about −1.25 V to +1.25 V) if `set_amp` sets peak-to-peak amplitude. That is not the 0 V to 2.5 V you set on the front panel. This cell uses offset = (high + low) / 2 instead. Confirm the printed settings show high ≈ 2.5 V and low ≈ 0 V.

In [ ]:
# This uses NaluInstruments, which is internal to Nalu Scientific.
# Use the appropriate instrument control code for your signal generator.
from naluinstruments.Instruments import Siglent_FuncGen

CHANNEL_INPUT = Siglent_FuncGen(1, "192.168.1.172")
TRIGGER_INPUT = Siglent_FuncGen(2, "192.168.1.172")

# Sine into the signal channel
CHANNEL_INPUT.set_burst("OFF")
CHANNEL_INPUT.set_load("50")
CHANNEL_INPUT.set_wavetype("SINE")
CHANNEL_INPUT.set_offset(0)
CHANNEL_INPUT.set_freq(SINE_FREQ)
CHANNEL_INPUT.set_amp(SINE_AMP)

# Bursted pulse into the external trigger input
TRIGGER_INPUT.set_burst("OFF")
TRIGGER_INPUT.set_load("50")
TRIGGER_INPUT.set_wavetype("PULSE")
TRIGGER_INPUT.set_freq(TRIGGER_FREQ)
TRIGGER_INPUT.set_amp(TRIGGER_HIGH - TRIGGER_LOW)            # Vpp
TRIGGER_INPUT.set_offset((TRIGGER_HIGH + TRIGGER_LOW) / 2)   # CHANGED (debug): was 0, which centers the pulse on 0 V
TRIGGER_INPUT.set_width(PULSE_WIDTH)
TRIGGER_INPUT.set_burst("ON")
TRIGGER_INPUT.set_burst_mode("NCYC")
TRIGGER_INPUT.set_burst_time(N_CYCLES)
TRIGGER_INPUT.set_burst_period(BURST_PERIOD)
TRIGGER_INPUT.set_delay(0)
TRIGGER_INPUT.set_burst_triggersource("INT")

# CHANGED (debug): make sure both outputs start off; the tests switch them on
CHANNEL_INPUT.rf_off()
TRIGGER_INPUT.rf_off()

print(CHANNEL_INPUT.get_output_settings())
print(TRIGGER_INPUT.get_output_settings())

## Data Capture Configuration

In [ ]:
DAC_LEVEL = 1804  # Default external DAC counts for HDSoCv2 (naluconfigs ext_dac default). Adjust for your board.

# You may have to adjust these to center the pulse in the captured data.
# HDSoCv2 has 62 windows of 32 samples.
WINDOWS = 62
LOOKBACK = 62
WAT = 31
READOUTCHANNELS = 60  # digital register 'readoutchannels' (8 bits on v2); 60 is the chip default

ENABLED_READOUT_CHANNELS = [*range(BOARD.channels)]  # 64 on v2

# CHANGED (debug): was 1 (one 1.2 s wait). With ~1 accepted trigger per burst and
# a 1 s burst period, a 1 s run can end before any trigger lands.
READOUT_TIME = 10        # s, for the external-trigger tests
READOUT_TIME_IMM = 2     # s, immediate trigger reads out continuously, so keep it short

TRIGGER_THRESHOLD = 100  # counts, on SIGNAL_CHANNEL; used by self-trigger / ROI

# CHANGED (debug): three readout configurations, one per test
READOUT_PARAMS_IMM_FORCED = {
    "trig": "imm",
    "lb": "forced",
    "acq": "raw",
    "ped": "zero",
    "readoutEn": True,
    "singleEv": False,
}
READOUT_PARAMS_EXT_FORCED = {**READOUT_PARAMS_IMM_FORCED, "trig": "ext"}
READOUT_PARAMS_EXT_ROI = {**READOUT_PARAMS_IMM_FORCED, "trig": "ext", "lb": "roi"}

READ_WINDOW = {
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
}

# Saved with each acquisition's metadata for later analysis.
PARAMS = {
    "board_model": BOARD_MODEL,
    "dac_level": DAC_LEVEL,
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
    "readoutchannels": READOUTCHANNELS,
    "signal_channel": SIGNAL_CHANNEL,
    "trigger_threshold": TRIGGER_THRESHOLD,
    "sine_freq": SINE_FREQ,
    "sine_amp": SINE_AMP,
    "trigger_freq": TRIGGER_FREQ,  # CHANGED: replaces undefined PULSE_FREQ (NameError in the v1 notebook)
    "trigger_high": TRIGGER_HIGH,
    "trigger_low": TRIGGER_LOW,
    "n_cycle": N_CYCLES,
    "burst_period": BURST_PERIOD,
    "pulse_width": PULSE_WIDTH,
    "readout_time": READOUT_TIME,
    "readout_channels": ENABLED_READOUT_CHANNELS,
}

AR = AnalogRegisters(BOARD)
CR = ControlRegisters(BOARD)
DR = DigitalRegisters(BOARD)
AM = AcquisitionManager(BOARD)

### Set Board Configurations

`exttrig` (control register 0x04, bit 10) is described in the v2 register map as the *HDSoC evtTrig_DEBUG_GCCClr pin; external trigger*. It is a software-driven level on that chip pin, not an enable for the SMA trigger input. Its default is 0 on v2 (it was 1 on v1), so writing 0 keeps the v2 default.

In [ ]:
BOARD.ext_bias.set_dacs(DAC_LEVEL)
CR.write("exttrig", 0)

### Generate / Load Pedestals

In [ ]:
PC = get_pedestals_controller(BOARD, num_captures=10, num_warmup_events=10)
PC.generate_pedestals()

### Setup Triggers

These settings matter for self-trigger and possibly for ROI channel selection. They should not decide when an event fires with `trig="ext"`.

On v2, TSEL and TSGN are per-channel bits in the `ch_fwd_<ch>` analog registers, so the v1 registers `tsel_left`, `tsel_right`, `tsgn_left` and `tsgn_right` don't exist. `write_triggers()` sets TSEL on all channels, and `BOARD.trigger.edges` sets TSGN.

In [ ]:
trigger_values = [0] * BOARD.channels
trigger_values[SIGNAL_CHANNEL] = TRIGGER_THRESHOLD  # CHANGED (debug): was channel 8
BOARD.trigger.values = trigger_values
BOARD.trigger.write_triggers()  # also sets TSEL on every channel (v2 controller)

# CHANGED (v2): four reference groups on v2 (v1 had two)
BOARD.trigger.references = {
    "0_15": (6, 10),
    "16_31": (6, 10),
    "32_47": (6, 10),
    "48_63": (6, 10),
}

# CHANGED (v2): replaces AR.write("tsgn_left"/"tsgn_right", 1).
# Sets the TSGN bit to 1 on every channel, matching the v1 notebook. Check the polarity for your input.
BOARD.trigger.edges = True

### Set Readout Settings

Note: in naludaq 0.36.1, `HDSoCv2ReadoutController.set_readout_channels` assigns to `activate_channels` instead of calling it. Channel activation is skipped (channels stay in their default active state), and the board controller's `activate_channels` method is replaced by a list for the rest of the session. The channel masks (`chanmask0`–`chanmask7`) are still written.

In [ ]:
BOARD.readout.set_readout_channels(ENABLED_READOUT_CHANNELS)
BOARD.readout.set_read_window(**READ_WINDOW)  # CHANGED: the duplicate positional call was removed
DR.write("readoutchannels", READOUTCHANNELS)
# CHANGED: acquisition creation was removed from this cell; each test below creates its own.

## Readout Tests

Run in order. Each test writes to its own acquisition, stored in `RESULTS`.

| Test 1 | Test 2 | Test 3 | Likely problem |
|---|---|---|---|
| 0 | – | – | Acquisition backend / output, not triggering |
| > 0 | 0 | – | External trigger path: cable, levels, termination (scope the input at the board) |
| > 0 | > 0 | 0 | ROI channel selection: thresholds on the signal channel |

In [ ]:
RESULTS = {}

### Test 1: immediate trigger, forced lookback

In [ ]:
RESULTS["imm_forced"] = run_readout_test(
    "imm_forced", READOUT_PARAMS_IMM_FORCED, use_ext_trigger=False, readout_time=READOUT_TIME_IMM
)

### Test 2: external trigger, forced lookback

Expect about 1 event per second. The chip is busy for much longer than the 1 µs pulse spacing, so the 2nd and 3rd pulses in each burst should be missed.

In [ ]:
RESULTS["ext_forced"] = run_readout_test(
    "ext_forced", READOUT_PARAMS_EXT_FORCED, use_ext_trigger=True, readout_time=READOUT_TIME
)

### Test 3: external trigger, ROI

In [ ]:
RESULTS["ext_roi"] = run_readout_test(
    "ext_roi", READOUT_PARAMS_EXT_ROI, use_ext_trigger=True, readout_time=READOUT_TIME
)

## Plot an Event

In [ ]:
PLOT_TEST = "ext_roi"  # one of: "imm_forced", "ext_forced", "ext_roi"
EVENT_IDX = 0          # CHANGED (debug): was 2
CHANNEL = SIGNAL_CHANNEL  # CHANGED (debug): was 8

acq = RESULTS[PLOT_TEST]
if len(acq) <= EVENT_IDX:
    print(f"{PLOT_TEST} has {len(acq)} events; nothing to plot at index {EVENT_IDX}.")
else:
    parser = get_parser(BOARD.params)
    correcter = PedestalsCorrecter(BOARD.params, BOARD.pedestals)
    event = correcter.run(acq.parsed_event(parser, EVENT_IDX))

    chan_data = event["data"][CHANNEL]
    labels = event["window_labels"][CHANNEL]
    print(f"Channel {CHANNEL}: {len(chan_data)} samples, window labels {list(labels)}")

    fig = create_fig(figsize=(8, 4))
    plot_single_event(event, show=False)
    plt.grid(True, axis="x", which="both")
    if len(chan_data) > 0:
        plt.xlim([0, len(chan_data)])
        ticks = [*range(0, len(chan_data), 32)]
        plt.xticks(ticks, labels=labels[: len(ticks)])
        plt.xlabel("Window Labels")
    plt.title(
        f"{PLOT_TEST} | WIN: {WINDOWS}, LB: {LOOKBACK}, WAT: {WAT}, "
        f"READOUT: {READOUTCHANNELS}, Pulse Width: {PULSE_WIDTH}, Event: {EVENT_IDX}"
    )
    plt.savefig(FIGURES_DIR / f"ROI_exttrig_{PLOT_TEST}.png")
    plt.show()